# Aegis evaluation lab — deterministic baseline

This suite evaluates:
- request validation;
- clarification fields;
- evidence-reference validation;
- selected numeric tool outputs.

It does not yet evaluate LLM-generated research or final reports.

Expected numeric values were checked independently:
- 12 / 1800 * 90 = 0.6
- 0.8 * 0.75 + 0.4 * 0.25 = 0.7
- 2 * 20 + 100 * 2 = 240

In [3]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
from pathlib import Path
import sys


# evals/ is a repository package, separate from installed src/aegis/.
# Support launching this notebook from either root or notebooks/.
current = Path.cwd().resolve()

project_root = next(
    (
        candidate
        for candidate in (current, *current.parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "evals" / "runner.py").is_file()
    ),
    None,
)

if project_root is None:
    raise RuntimeError("Could not locate the Aegis project root")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import only after the project root is on sys.path.
from evals.runner import DATASET_PATH, parse_cases, print_report, run_suite


In [5]:
# Run the same evaluator used by the terminal command.
# Each invocation saves its own result file.
report, result_path = run_suite()

print_report(report)
print("Results saved to:", result_path)

expected_total = len(parse_cases(DATASET_PATH.read_text(encoding="utf-8")))
assert report["total"] == expected_total
assert report["failed"] == 0

CASE                       CATEGORY        RESULT
------------------------------------------------
football_research          valid           PASS
football_compare           valid           PASS
football_rank              valid           PASS
platform_research          valid           PASS
platform_compare           valid           PASS
platform_rank              valid           PASS
missing_factors            clarification   PASS
missing_period             clarification   PASS
missing_coverage           clarification   PASS
missing_ranking_basis      clarification   PASS
compare_one_subject        invalid         PASS
duplicate_subjects         invalid         PASS
empty_objective            invalid         PASS
unknown_source             reference       PASS
unknown_evidence           reference       PASS
d3_player_a_scoring        valid           PASS
d3_player_a_creativity     valid           PASS
d3_player_a_availability   valid           PASS
d3_player_b_scoring        valid     

In [6]:
# Inspect one result to understand exactly what was compared.
import json

example = next(
    result
    for result in report["results"]
    if result["id"] == "football_research"
)

print(json.dumps(example, indent=2))

{
  "id": "football_research",
  "category": "valid",
  "passed": true,
  "expected": {
    "outcome": "accepted",
    "clarification_fields": [],
    "error_type": null,
    "error_contains": null,
    "tool_outputs": [
      0.6
    ],
    "workers": [],
    "tool_sequence": []
  },
  "actual": {
    "outcome": "accepted",
    "clarification_fields": [],
    "tool_outputs": [
      0.6
    ],
    "workers": [
      [
        "evidence",
        "Player A",
        "Scoring"
      ],
      [
        "analysis",
        "Player A",
        "Scoring"
      ]
    ],
    "errors": []
  },
  "failures": []
}


In [7]:
import json

from evals.runner import DATASET_PATH, parse_cases

# Read and preserve the original cases.
existing_text = DATASET_PATH.read_text(encoding="utf-8")
existing_cases = parse_cases(existing_text)
existing_by_id = {case.id: case for case in existing_cases}

new_records = []

# Nine football cases: three players × three factors.
football_factors = {
    "scoring": (
        "Report goals per 90 for 2025. Use per_90_tool with goals "
        "and minutes from the evidence.",
        ["lookup_evidence_tool", "per_90_tool"],
    ),
    "creativity": (
        "Report assists and key passes for 2025 directly from evidence. "
        "Do not calculate derived metrics.",
        ["lookup_evidence_tool"],
    ),
    "availability": (
        "Report matches and minutes for 2025 directly from evidence. "
        "Do not calculate derived metrics.",
        ["lookup_evidence_tool"],
    ),
}

for letter in ("A", "B", "C"):
    subject = f"Player {letter}"

    for factor, (instruction, tools) in football_factors.items():
        new_records.append({
            "id": f"d3_player_{letter.lower()}_{factor}",
            "category": "valid",
            "input": {
                "request": {
                    "objective": f"Research {subject}. {instruction}",
                    "subjects": [subject],
                    "factors": [factor],
                    "mode": "research",
                    "scope": {
                        "time_period": "2025",
                        "coverage": "Demo League",
                    },
                },
                "dataset_ref": "football_demo_v1",
            },
            "expected": {
                "outcome": "accepted",
                "workers": [
                    ["evidence", subject, factor],
                    ["analysis", subject, factor],
                ],
                "tool_sequence": tools,
            },
        })

# Six platform cases: three vendors × two factors.
platform_factors = {
    "operating cost": (
        "Calculate the total price for 2 TB of storage for one month "
        "and 100 compute hours. Use calculator_tool for "
        "(2 * storage_usd_tb_month) + (100 * compute_usd_hour).",
        ["lookup_evidence_tool", "calculator_tool"],
    ),
    "reliability": (
        "Report the SLA percentage directly from evidence. "
        "Do not calculate derived metrics.",
        ["lookup_evidence_tool"],
    ),
}

for letter in ("A", "B", "C"):
    subject = f"Vendor {letter}"

    for factor, (instruction, tools) in platform_factors.items():
        new_records.append({
            "id": f"d3_vendor_{letter.lower()}_{factor.replace(' ', '_')}",
            "category": "valid",
            "input": {
                "request": {
                    "objective": f"Research {subject}. {instruction}",
                    "subjects": [subject],
                    "factors": [factor],
                    "mode": "research",
                    "scope": {
                        "time_period": "Fictional v1 snapshot",
                        "coverage": (
                            "Fictional platform specifications; "
                            "2 TB-month storage and 100 compute hours"
                        ),
                    },
                },
                "dataset_ref": "platforms_demo_v1",
            },
            "expected": {
                "outcome": "accepted",
                "workers": [
                    ["evidence", subject, factor],
                    ["analysis", subject, factor],
                ],
                "tool_sequence": tools,
            },
        })

assert len(new_records) == 15

# Validate definitions before changing the dataset.
new_cases = parse_cases(
    "\n".join(json.dumps(record) for record in new_records)
)

# Make rerunning this cell safe: don't append duplicate IDs.
for case in new_cases:
    if case.id in existing_by_id:
        if existing_by_id[case.id].model_dump() != case.model_dump():
            raise ValueError(f"Existing case differs: {case.id}")
    else:
        existing_cases.append(case)

assert len(existing_cases) == 30

DATASET_PATH.write_text(
    "\n".join(case.model_dump_json() for case in existing_cases) + "\n",
    encoding="utf-8",
)

print("PASS: 30 validated cases saved")

PASS: 30 validated cases saved


In [8]:
from evals.evaluators import routing_matches
from evals.runner import run_suite, print_report

offline_report, offline_path = run_suite()
print_report(offline_report)

# Only the 15 explicitly labelled routing cases form this denominator.
routing_cases = [
    result
    for result in offline_report["results"]
    if result["expected"].get("workers")
]

routing_passes = sum(
    result["actual"].get("outcome") == "accepted"
    and routing_matches(
        result["expected"]["workers"],
        result["actual"].get("workers", []),
    )
    for result in routing_cases
)

assert len(routing_cases) == 15

print(f"Routing accuracy: {routing_passes}/15")
print("Saved:", offline_path)

CASE                       CATEGORY        RESULT
------------------------------------------------
football_research          valid           PASS
football_compare           valid           PASS
football_rank              valid           PASS
platform_research          valid           PASS
platform_compare           valid           PASS
platform_rank              valid           PASS
missing_factors            clarification   PASS
missing_period             clarification   PASS
missing_coverage           clarification   PASS
missing_ranking_basis      clarification   PASS
compare_one_subject        invalid         PASS
duplicate_subjects         invalid         PASS
empty_objective            invalid         PASS
unknown_source             reference       PASS
unknown_evidence           reference       PASS
d3_player_a_scoring        valid           PASS
d3_player_a_creativity     valid           PASS
d3_player_a_availability   valid           PASS
d3_player_b_scoring        valid     

In [ ]:
import hashlib
from collections import Counter
from datetime import datetime, timezone
from decimal import Decimal

from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

from aegis.config import get_settings
from aegis.graphs.baseline import (
    BASELINE_SYSTEM_PROMPT,
    BASELINE_TOOLS,
    bundle_from_messages,
    requested_tool_calls,
)
from aegis.llm.budget import BudgetExceededError
from aegis.llm.callbacks import new_run_budget
from aegis.llm.gateway import get_model
from aegis.schemas import (
    FinalReport,
    ResearchRequest,
    validate_report_against_bundle,
)
from evals.evaluators import tool_metrics
from evals.runner import DATASET_PATH, RESULTS_DIRECTORY, parse_cases

# Freeze the dataset snapshot used by this experiment.
dataset_bytes = DATASET_PATH.read_bytes()
dataset_hash = hashlib.sha256(dataset_bytes).hexdigest()

cases_by_id = {
    case.id: case
    for case in parse_cases(dataset_bytes.decode("utf-8"))
}

live_ids = [
    "d3_player_a_scoring",
    "d3_player_b_scoring",
    "d3_player_c_scoring",
    "d3_vendor_a_operating_cost",
    "d3_vendor_b_operating_cost",
]

settings = get_settings()
live_settings = settings.model_copy(
    update={
        "budget_limit_usd": min(
            settings.budget_limit_usd,
            Decimal("0.10"),
        )
    }
)

# One tracker covers all five cases.
tracker, base_config = new_run_budget(live_settings)

agent = create_agent(
    model=get_model("researcher"),
    tools=BASELINE_TOOLS,
    system_prompt=BASELINE_SYSTEM_PROMPT,
    response_format=ToolStrategy(
        FinalReport,
        handle_errors=False,
    ),
)

started = datetime.now(timezone.utc)
live_results = []

RESULTS_DIRECTORY.mkdir(parents=True, exist_ok=True)
live_path = RESULTS_DIRECTORY / (
    started.strftime("day3_live_%Y%m%dT%H%M%S_%fZ.json")
)


def save_live_progress():
    """Checkpoint after each case, including failures and spending."""

    live_path.write_text(
        json.dumps(
            {
                "experiment": "A",
                "dataset_sha256": dataset_hash,
                "started_at": started.isoformat(),
                "selected_cases": live_ids,
                "budget_cutoff_usd": str(live_settings.budget_limit_usd),
                "recorded_spend_usd": str(tracker.spent_usd),
                "results": live_results,
            },
            indent=2,
            allow_nan=False,
        ),
        encoding="utf-8",
    )